# FoodMarket AI : J2 — Qualité des données alimentaires
**Objectif :** identifier les anomalies, nettoyer les données et produire un fichier fiable pour les futures analyses.

In [1]:

%pip install duckdb pyarrow pandas requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

In [3]:

from pathlib import Path
import pandas as pd
import duckdb
import requests



## 1. Vérifier le fichier Parquet

In [4]:

racine = Path(r"D:\Projets_Data\FoodMarket_AI")

fichier_prix = racine / "data" / "raw" / "prices.parquet"

print("Fichier trouvé :", fichier_prix.exists())

if fichier_prix.exists():
    print(
        "Taille (Mo) :",
        round(fichier_prix.stat().st_size / 1024**2, 2)
    )


Fichier trouvé : True
Taille (Mo) : 33.26


In [5]:
#Inspecter la structure


con = duckdb.connect()

con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 2")

schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(fichier_prix)]
).df()

display(schema)


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,type,VARCHAR,YES,None,None,None
2,product_code,VARCHAR,YES,None,None,None
3,product_name,VARCHAR,YES,None,None,None
4,category_tag,VARCHAR,YES,None,None,None
5,labels_tags,VARCHAR[],YES,None,None,None
6,origins_tags,VARCHAR[],YES,None,None,None
7,price,"DECIMAL(10,3)",YES,None,None,None
8,price_is_discounted,BOOLEAN,YES,None,None,None
9,price_without_discount,"DECIMAL(10,3)",YES,None,None,None


In [6]:
#Lire les 10 observations


echantillon = con.execute(
    "SELECT * FROM read_parquet(?) LIMIT 10",
    [str(fichier_prix)]
).df()

display(echantillon)


,id,type,product_code,product_name,category_tag,labels_tags,origins_tags,price,price_is_discounted,price_without_discount,...,location_osm_address_postcode,location_osm_address_city,location_osm_address_country,location_osm_address_country_code,location_osm_lat,location_osm_lon,location_website_url,location_source,location_created,location_updated
0,42629,PRODUCT,3560070283484,None,None,<NA>,<NA>,1.25,False,NaN,...,75012,Paris,France,FR,48.848500,2.370951,None,None,2024-01-16 19:29:47.461589+01:00,2026-09-16 11:11:34.636384+02:00
1,42630,PRODUCT,3560070206469,None,None,<NA>,<NA>,1.99,False,NaN,...,75012,Paris,France,FR,48.848500,2.370951,None,None,2024-01-16 19:29:47.461589+01:00,2026-09-16 11:11:34.636384+02:00
2,39637,PRODUCT,3123930711758,None,None,<NA>,<NA>,3.60,False,NaN,...,17470,Aulnay,France,FR,46.031470,-0.347739,None,None,2024-10-19 10:12:54.613170+02:00,2026-09-16 11:42:37.088580+02:00
3,39573,PRODUCT,3500390010247,None,None,<NA>,<NA>,19.95,False,NaN,...,38100,Grenoble,France,FR,45.176950,5.728508,None,None,2023-12-24 11:09:56.269605+01:00,2026-09-16 12:06:08.691054+02:00
4,33784,PRODUCT,8033976484743,None,None,<NA>,<NA>,2.99,False,NaN,...,91026,Mazara del Vallo,Italia,IT,37.653750,12.603567,None,None,2024-08-26 15:54:56.009100+02:00,2024-08-26 19:59:56.261823+02:00
5,39518,PRODUCT,3760020507916,None,None,<NA>,<NA>,3.13,False,NaN,...,38100,Grenoble,France,FR,45.179382,5.726661,None,None,2023-12-19 20:11:02.694100+01:00,2024-09-01 15:28:19.350103+02:00
6,39625,PRODUCT,3220035229611,None,None,<NA>,<NA>,2.77,False,NaN,...,17470,Aulnay,France,FR,46.031470,-0.347739,None,None,2024-10-19 10:12:54.613170+02:00,2026-09-16 11:42:37.088580+02:00
7,39622,PRODUCT,3770014111777,None,None,<NA>,<NA>,2.04,False,NaN,...,17470,Aulnay,France,FR,46.031470,-0.347739,None,None,2024-10-19 10:12:54.613170+02:00,2026-09-16 11:42:37.088580+02:00
8,39626,PRODUCT,3250392335137,None,None,<NA>,<NA>,2.07,False,NaN,...,17470,Aulnay,France,FR,46.031470,-0.347739,None,None,2024-10-19 10:12:54.613170+02:00,2026-09-16 11:42:37.088580+02:00
9,39610,PRODUCT,3280220908073,None,None,<NA>,<NA>,1.48,False,NaN,...,17470,Aulnay,France,FR,46.031470,-0.347739,None,None,2024-10-19 10:12:54.613170+02:00,2026-09-16 11:42:37.088580+02:00


## 2. Identifier les colonnes réellement disponibles

In [7]:

colonnes = schema["column_name"].tolist()

print(colonnes)


['id', 'type', 'product_code', 'product_name', 'category_tag', 'labels_tags', 'origins_tags', 'price', 'price_is_discounted', 'price_without_discount', 'discount_type', 'price_per', 'currency', 'location_osm_id', 'location_osm_type', 'location_id', 'date', 'proof_id', 'receipt_quantity', 'owner', 'source', 'created', 'updated', 'proof_file_path', 'proof_mimetype', 'proof_type', 'proof_date', 'proof_currency', 'proof_receipt_price_count', 'proof_receipt_price_total', 'proof_owner', 'proof_source', 'proof_created', 'proof_updated', 'location_type', 'location_osm_display_name', 'location_osm_tag_key', 'location_osm_tag_value', 'location_osm_address_postcode', 'location_osm_address_city', 'location_osm_address_country', 'location_osm_address_country_code', 'location_osm_lat', 'location_osm_lon', 'location_website_url', 'location_source', 'location_created', 'location_updated']


In [8]:
#Compter les observations


resultat = con.execute(
    "SELECT COUNT(*) AS nb_lignes FROM read_parquet(?)",
    [str(fichier_prix)]
).df()

display(resultat)


,nb_lignes
0,339261


## 3. Contrôler la qualité des prix

In [9]:
#Vérifier les champs nécessaires

colonnes_requises = [
    "product_code",
    "price",
    "date"
]

manquantes = [
    c for c in colonnes_requises
    if c not in colonnes
]

print("Colonnes manquantes :", manquantes)


Colonnes manquantes : []


In [10]:
#Statistiques de qualité


qualite = con.execute("""
    SELECT
        COUNT(*) AS nb_observations,
        COUNT(DISTINCT product_code) AS nb_produits,
        COUNT(*) FILTER (
            WHERE product_code IS NULL
        ) AS codes_manquants,
        COUNT(*) FILTER (
            WHERE price IS NULL
        ) AS prix_manquants,
        COUNT(*) FILTER (
            WHERE TRY_CAST(price AS DOUBLE) <= 0
        ) AS prix_non_positifs,
        COUNT(*) FILTER (
            WHERE TRY_CAST(date AS DATE) IS NULL
        ) AS dates_invalides
    FROM read_parquet(?)
""", [str(fichier_prix)]).df()

display(qualite)


,nb_observations,nb_produits,codes_manquants,prix_manquants,prix_non_positifs,dates_invalides
0,339261,148300,10556,0,13,185


In [11]:
# Examiner les prix


statistiques = con.execute("""
    SELECT
        MIN(TRY_CAST(price AS DOUBLE)) AS prix_min,
        AVG(TRY_CAST(price AS DOUBLE)) AS prix_moyen,
        MEDIAN(TRY_CAST(price AS DOUBLE)) AS prix_median,
        MAX(TRY_CAST(price AS DOUBLE)) AS prix_max
    FROM read_parquet(?)
    WHERE TRY_CAST(price AS DOUBLE) > 0
""", [str(fichier_prix)]).df()

display(statistiques)


,prix_min,prix_moyen,prix_median,prix_max
0,0.01,24.779249,3.35,604000.0


## 4. Construire un premier fichier propre

In [12]:

dossier_sortie = racine / "data" / "processed"
dossier_sortie.mkdir(parents=True, exist_ok=True)

sortie = dossier_sortie / "prix_valides_j2.parquet"

con.execute("""
    COPY (
        SELECT
            CAST(product_code AS VARCHAR) AS product_code,
            TRY_CAST(price AS DOUBLE) AS price,
            TRY_CAST(date AS DATE) AS date
        FROM read_parquet(?)
        WHERE product_code IS NOT NULL
          AND TRY_CAST(price AS DOUBLE) > 0
          AND TRY_CAST(date AS DATE) IS NOT NULL
    )
    TO ?
    (FORMAT PARQUET)
""", [str(fichier_prix), str(sortie)])

print("Fichier créé :", sortie)


Fichier créé : D:\Projets_Data\FoodMarket_AI\data\processed\prix_valides_j2.parquet


## 5. Enrichir les prix avec Open Data Facts

In [13]:

codes = con.execute("""
    SELECT DISTINCT product_code
    FROM read_parquet(?)
    WHERE product_code IS NOT NULL
      AND LENGTH(CAST(product_code AS VARCHAR)) BETWEEN 8 AND 14
    LIMIT 5
""", [str(sortie)]).df()

display(codes)


,product_code
0,3560070283484
1,3560070206469
2,3123930711758
3,3500390010247
4,8033976484743


## 6. Récupérer les informations produits

In [14]:
url = "https://world.openfoodfacts.org/api/v2/search"

params = {
    "categories_tags_en": "breakfast-cereals",
    "page_size": 20,
    "fields": "code,product_name,brands,categories_tags"
}

headers = {
    "User-Agent": "FoodMarketAI/0.1 (educational project)"
}

try:
    reponse = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )
    reponse.raise_for_status()

    produits = reponse.json().get("products", [])
    df_produits = pd.DataFrame(produits)

except requests.exceptions.RequestException as erreur:
    print("Erreur API :", erreur)
    df_produits = pd.DataFrame()


In [15]:
#Vérifier et filtrer la catégorie


if not df_produits.empty:

    df_cereales = df_produits[
        df_produits["categories_tags"].apply(
            lambda categories:
                isinstance(categories, list)
                and "en:breakfast-cereals" in categories
        )
    ].copy()

    print("Nombre de céréales :", len(df_cereales))

    display(df_cereales.head())

else:
    print("Aucune donnée produit disponible.")



Nombre de céréales : 20


,brands,categories_tags,code,product_name
0,"Cruesli, Quaker","[en:plant-based-foods-and-beverages, en:plant-...",3168930010265,cruesly mélange de noix
1,Bjorg,"[en:plant-based-foods-and-beverages, en:plant-...",3229820160672,Croustillant Chocolat
2,Bjorg,"[en:plant-based-foods-and-beverages, en:plant-...",3229820019307,Flocons d'avoine
3,Bjorg,"[en:plant-based-foods-and-beverages, en:plant-...",3229820129488,"muesli raisin,datte,figue"
4,"Nestlé, Chocapic","[en:plant-based-foods-and-beverages, en:plant-...",3387390123210,Chocapic


## 7. Filtrer la jointure avec Open Data Prices

In [16]:


import duckdb
from pathlib import Path

racine = Path(r"D:\Projets_Data\FoodMarket_AI")

fichier_prix = racine / "data/raw/prices.parquet"

con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 2")

if not df_cereales.empty:

    codes = df_cereales[["code"]].drop_duplicates().copy()
    codes["code"] = codes["code"].astype("string")

    con.register("codes_cereales", codes)

    df_prix_cereales = con.execute("""
        SELECT
            p.*
        FROM read_parquet(?) AS p
        INNER JOIN codes_cereales AS c
            ON CAST(p.product_code AS VARCHAR) = c.code
    """, [str(fichier_prix)]).df()

    print("Observations de prix des céréales :",
          len(df_prix_cereales))

    display(df_prix_cereales.head())


Observations de prix des céréales : 385


,id,type,product_code,product_name,category_tag,labels_tags,origins_tags,price,price_is_discounted,price_without_discount,...,location_osm_address_postcode,location_osm_address_city,location_osm_address_country,location_osm_address_country_code,location_osm_lat,location_osm_lon,location_website_url,location_source,location_created,location_updated
0,2445,PRODUCT,3229820782560,None,None,<NA>,<NA>,4.19,False,NaN,...,38000,Grenoble,France,FR,45.183380,5.722133,None,None,2023-12-24 10:02:10.300559+01:00,2026-09-16 11:27:53.362214+02:00
1,2443,PRODUCT,3229820129488,None,None,<NA>,<NA>,2.75,False,NaN,...,38000,Grenoble,France,FR,45.183380,5.722133,None,None,2023-12-24 10:02:10.300559+01:00,2026-09-16 11:27:53.362214+02:00
2,6038,PRODUCT,7613036669146,None,None,<NA>,<NA>,4.57,False,NaN,...,38100,Grenoble,France,FR,45.179382,5.726661,None,None,2023-12-19 20:11:02.694100+01:00,2024-09-01 15:28:19.350103+02:00
3,6045,PRODUCT,5010477348630,None,None,<NA>,<NA>,3.89,False,NaN,...,38100,Grenoble,France,FR,45.179382,5.726661,None,None,2023-12-19 20:11:02.694100+01:00,2024-09-01 15:28:19.350103+02:00
4,6057,PRODUCT,5010029000023,None,None,<NA>,<NA>,3.91,False,NaN,...,38100,Grenoble,France,FR,45.179382,5.726661,None,None,2023-12-19 20:11:02.694100+01:00,2024-09-01 15:28:19.350103+02:00


In [17]:
#Enregistrer les résultats


if not df_cereales.empty:

    sortie = racine / "data/processed"
    sortie.mkdir(parents=True, exist_ok=True)

    df_cereales.to_parquet(
        sortie / "cereales_produits_j2.parquet",
        index=False
    )

    df_prix_cereales.to_parquet(
        sortie / "cereales_prix_j2.parquet",
        index=False
    )

    print("Fichiers enregistrés.")


Fichiers enregistrés.
